# SEXTANTE — análisis reproducible (Fase 3)

Run fijado: `20261124T113018928192Z-dff10b9e`  
Revisión de datos: `dff10b9e08629137bd70864b64681385ab8eff72`  
Procesado: 2026-11-24 12:07 UTC

## De dónde salen las cifras

**Todos los valores están copiados de los artefactos derivados de ese run; este notebook NO recalcula TF-IDF, NMF ni K-Means desde el corpus crudo.** La fuente es el repositorio privado [pxtron/vacantes-colombia](https://huggingface.co/datasets/pxtron/vacantes-colombia), en `derived/runs/<run>/`:

- [Carpeta del run en `main`](https://huggingface.co/datasets/pxtron/vacantes-colombia/tree/main/derived/runs/20261124T113018928192Z-dff10b9e) — contiene `manifest.json`, `eda.json`, `segment_profiles.csv`, `clustering_comparison.json`, `supervised_metrics.json`, `graph_metrics.json` y `skill_graph_nodes.parquet`.
- [Carpeta en la revisión fijada](https://huggingface.co/datasets/pxtron/vacantes-colombia/tree/dff10b9e08629137bd70864b64681385ab8eff72/derived/runs/20261124T113018928192Z-dff10b9e) — la misma, pero anclada al commit exacto de las fuentes.
- `manifest.json` es el índice: guarda `source.rows`, `source.revision`, todos los resultados y el sha256 de cada artefacto. `derived/latest.json` es el puntero que lee la API web; ahora apunta a este run.

| Valor del notebook | Artefacto | Campo |
| --- | --- | --- |
| `rows`, `text_rows`, `repaired_text_cells`, `skill_mentions`, `duplicate_*`, `clusters`, `topics`, `tfidf_features`, `silhouette_full` | `manifest.json` | `source.rows`, `results.*` |
| `missing_salary_min/max`, `empty_descriptions`, `sources` | `eda.json` | `missing_fraction.*`, `description.empty_rows`, `top_source` |
| `comparison`, `k_sweep` | `clustering_comparison.json` | `methods.*`, `kmeans_k_sweep_silhouette`, `sample_rows` |
| `segments` | `segment_profiles.csv` | conteo por clúster |
| `salary` | `supervised_metrics.json` | `classification.*`, `labeled_rows`, `labels` |
| `graph` | `graph_metrics.json` / `manifest.json` | `results.graph.*` |
| `top_weighted` | `skill_graph_nodes.parquet` | `skill_label`, `weighted_degree`, `postings` |

La celda siguiente imprime este mapa de forma ejecutable.

## Por qué están copiados y qué NO prueba este notebook

- Recalcular el run completo exige unos 2,5 GB de RAM (TF-IDF sobre 50.000 términos y K-Means); el notebook anterior del repo corría sobre 48 filas **sintéticas** y dejaba una silueta falsa de `0,9999998211860657`, que se descartó. Copiar valores reales evitó publicar un número inventado.
- Las aserciones al final son **circulares**: comprueban que `DATA` coincida con el paper, **no** que coincida con Hugging Face. El contraste con HF es manual, usando el enlace de arriba y `GRUPO_D_FASE_3_Datos.txt`.
- No se hizo revisión manual de muestra: no hay vacantes revisadas ni acuerdos de anotación que reportar.
- Si el run se regenera de nuevo en HF, `latest.json` cambiará pero este notebook seguirá mostrando las cifras del 2026-11-24.
- Los `docs/` del repo documentan el esquema general (`docs/data.md:82-83,164-165`, `docs/architecture.md:476-478,636`: `derived/runs/<run_id>/` y `derived/latest.json`); `docs/techniques.md` §10 define el grado ponderado del grafo.

El notebook verifica la coherencia interna de esas cifras y reproduce las del paper. No usa datos sintéticos.

In [1]:
DATA = {
  "run_id": "20261124T113018928192Z-dff10b9e",
  "revision": "dff10b9e08629137bd70864b64681385ab8eff72",
  "rows": 557279,
  "text_rows": 557279,
  "repaired_text_cells": 95879,
  "empty_descriptions": 626,
  "missing_salary_min": 0.22560692220593276,
  "missing_salary_max": 0.22734393364903396,
  "skill_mentions": 1131942,
  "duplicate_candidates": 415641,
  "duplicate_groups": 43014,
  "duplicate_postings": 215881,
  "clusters": 16,
  "topics": 12,
  "tfidf_features": 50000,
  "silhouette_full": 0.014982510358095169,
  "silhouette_sample_rows": 5000,
  "comparison": {
    "kmeans": 0.20097562670707703,
    "ward": 0.1464429497718811,
    "dbscan": -0.14483319222927094,
    "dbscan_noise": 0.27,
    "sample_rows": 3000
  },
  "k_sweep": {
    "12": 0.17248469591140747,
    "16": 0.1882595270872116,
    "20": 0.21632495522499084,
    "24": 0.21586421132087708,
    "8": 0.14634591341018677
  },
  "segments": [
    [
      100975,
      "Servicio al cliente"
    ],
    [
      88565,
      "Gestion/Analista"
    ],
    [
      62869,
      "Comercial"
    ],
    [
      43059,
      "Ventas"
    ],
    [
      34609,
      "Produccion"
    ],
    [
      31114,
      "Construccion"
    ],
    [
      28881,
      "Bodega"
    ],
    [
      28576,
      "Call center"
    ]
  ],
  "salary": {
    "accuracy": 0.74841,
    "baseline_within_one_band": 0.85577,
    "class_counts": [
      19127,
      261042,
      88334,
      37024,
      16863,
      9115
    ],
    "cv_folds": 5,
    "cv_rows": 100000,
    "macro_f1": 0.562928442372711,
    "model": "LogisticRegression(multinomial) over SPE salary bands; TF-IDF + one-hot categoricals",
    "within_one_band": 0.9448,
    "labeled_rows": 431505,
    "labels": [
      "Hasta $1,5M",
      "$1,5M – $2M",
      "$2M – $3M",
      "$3M – $4M",
      "$4M – $6M",
      "Más de $6M"
    ],
    "baseline_accuracy": 0.60418,
    "baseline_macro_f1": 0.12554285263083528,
    "bands": [
      "Hasta $1,5M",
      "$1,5M – $2M",
      "$2M – $3M",
      "$3M – $4M",
      "$4M – $6M",
      "Más de $6M"
    ]
  },
  "graph": {
    "skill_nodes": 400,
    "skill_edges": 24680,
    "communities": 6,
    "modularity": 0.27754119554592666,
    "segment_nodes": 16,
    "segment_edges": 35,
    "transition_routes": 240
  },
  "top_weighted": [
    {
      "skill": "comunicación",
      "weighted_degree": 174955,
      "postings": 71387,
      "degree": 398
    },
    {
      "skill": "servicio al cliente",
      "weighted_degree": 172934,
      "postings": 101043,
      "degree": 380
    },
    {
      "skill": "técnicas de venta",
      "weighted_degree": 166575,
      "postings": 107330,
      "degree": 354
    },
    {
      "skill": "trabajar en equipos",
      "weighted_degree": 113142,
      "postings": 51067,
      "degree": 398
    },
    {
      "skill": "utilizar software de hojas de cálculo",
      "weighted_degree": 101280,
      "postings": 34267,
      "degree": 376
    }
  ],
  "sources": {
    "bayt": 246,
    "elempleo": 2089,
    "indeed": 3054,
    "linkedin": 962,
    "spe": 550928
  }
}
print("Run:", DATA["run_id"])
print("Revision de datos:", DATA["revision"])
print("Artefactos: manifest, eda, segment_profiles, clustering_comparison, supervised_metrics, graph_metrics, skill_graph_nodes")

Run: 20261124T113018928192Z-dff10b9e
Revision de datos: dff10b9e08629137bd70864b64681385ab8eff72
Artefactos: manifest, eda, segment_profiles, clustering_comparison, supervised_metrics, graph_metrics, skill_graph_nodes


In [2]:
PROVENANCE = {
    "run_id / revision / rows": ("manifest.json", "run_id / source.revision / source.rows"),
    "repaired_text_cells": ("manifest.json", "results.repaired_text_cells"),
    "text_rows / clusters / topics / tfidf_features": ("manifest.json", "results.text_rows / cluster_count / topic_count / tfidf_features"),
    "skill_mentions": ("manifest.json", "results.skill_mentions"),
    "duplicate_candidates / groups / postings": ("manifest.json", "results.duplicate_candidates / groups / postings"),
    "silhouette_full / silhouette_sample_rows": ("manifest.json", "results.silhouette_cosine_sample / silhouette_sample_rows"),
    "empty_descriptions / missing_salary_* / sources": ("eda.json", "description.empty_rows / missing_fraction.* / top_source"),
    "comparison / k_sweep": ("clustering_comparison.json", "methods.*.silhouette_cosine / kmeans_k_sweep_silhouette / sample_rows"),
    "segments": ("segment_profiles.csv", "counts by cluster"),
    "salary": ("supervised_metrics.json", "classification.* / labeled_rows / labels"),
    "graph": ("graph_metrics.json", "skill_nodes / skill_edges / communities / modularity"),
    "top_weighted": ("skill_graph_nodes.parquet", "skill_label / weighted_degree / postings"),
}
print("Mapa de procedencia (valores copiados, no recalculados)")
print("-" * 52)
for key, (artifact, field) in PROVENANCE.items():
    print(f"{key:48s} <- {artifact}: {field}")
print("Indice y sha256 de cada artefacto: manifest.json del run fijado")

Mapa de procedencia (valores copiados, no recalculados)
----------------------------------------------------
run_id / revision / rows                         <- manifest.json: run_id / source.revision / source.rows
repaired_text_cells                              <- manifest.json: results.repaired_text_cells
text_rows / clusters / topics / tfidf_features   <- manifest.json: results.text_rows / cluster_count / topic_count / tfidf_features
skill_mentions                                   <- manifest.json: results.skill_mentions
duplicate_candidates / groups / postings         <- manifest.json: results.duplicate_candidates / groups / postings
silhouette_full / silhouette_sample_rows         <- manifest.json: results.silhouette_cosine_sample / silhouette_sample_rows
empty_descriptions / missing_salary_* / sources  <- eda.json: description.empty_rows / missing_fraction.* / top_source
comparison / k_sweep                             <- clustering_comparison.json: methods.*.silhouette_cosine 

In [3]:
print("Corpus y limpieza")
print("-" * 52)
print(f"Registros en la revision ............ {DATA['rows']:,}".replace(",", "."))
print(f"Registros con texto utilizable ...... {DATA['text_rows']:,}".replace(",", "."))
print(f"Celdas de texto reparadas ........... {DATA['repaired_text_cells']:,}".replace(",", "."))
print(f"Descripciones vacias ................ {DATA['empty_descriptions']}")
print(f"Sinsalario minimo ................... {DATA['missing_salary_min']:.1%}")
print(f"Sin salario maximo .................. {DATA['missing_salary_max']:.1%}")
print(f"Vacantes con banda oficial .......... {DATA['salary']['labeled_rows']:,}".replace(",", "."))
print("Fuentes:", ", ".join(f"{k}: {v:,}".replace(",", ".") for k, v in DATA["sources"].items()))

Corpus y limpieza
----------------------------------------------------
Registros en la revision ............ 557.279
Registros con texto utilizable ...... 557.279
Celdas de texto reparadas ........... 95.879
Descripciones vacias ................ 626
Sinsalario minimo ................... 22.6%
Sin salario maximo .................. 22.7%
Vacantes con banda oficial .......... 431.505
Fuentes: bayt: 246, elempleo: 2.089, indeed: 3.054, linkedin: 962, spe: 550.928


In [4]:
print("Habilidades ESCO y duplicados")
print("-" * 52)
print(f"Menciones de habilidades ............ {DATA['skill_mentions']:,}".replace(",", "."))
print(f"Pares candidatos a duplicado ........ {DATA['duplicate_candidates']:,}".replace(",", "."))
print(f"Grupos de duplicados ................ {DATA['duplicate_groups']:,}".replace(",", "."))
print(f"Vacantes en esos grupos ............. {DATA['duplicate_postings']:,}".replace(",", "."))
print("Los pares se revisan manualmente: no se eliminan vacantes de forma automatica.")

Habilidades ESCO y duplicados
----------------------------------------------------
Menciones de habilidades ............ 1.131.942
Pares candidatos a duplicado ........ 415.641
Grupos de duplicados ................ 43.014
Vacantes en esos grupos ............. 215.881
Los pares se revisan manualmente: no se eliminan vacantes de forma automatica.


In [5]:
print("Segmentacion ocupacional")
print("-" * 52)
print(f"Caracteristicas TF-IDF .............. {DATA['tfidf_features']:,}".replace(",", "."))
print(f"Temas NMF ........................... {DATA['topics']}")
print(f"Perfiles K-Means .................... {DATA['clusters']}")
print(f"Silueta coseno (muestra completa) ... {DATA['silhouette_full']:.4f} (n={DATA['silhouette_sample_rows']})")
print("Barrido de k:")
for k, s in sorted(DATA["k_sweep"].items()):
    print(f"  k={k} -> silueta={s:.4f}")
print("Comparacion de metodos (n=3.000):")
for name, s in DATA["comparison"].items():
    if isinstance(s, float):
        print(f"  {name:8s} -> silueta={s:.4f}")
print(f"  ruido DBSCAN -> {DATA['comparison']['dbscan_noise']:.1%}")

Segmentacion ocupacional
----------------------------------------------------
Caracteristicas TF-IDF .............. 50.000
Temas NMF ........................... 12
Perfiles K-Means .................... 16
Silueta coseno (muestra completa) ... 0.0150 (n=5000)
Barrido de k:
  k=12 -> silueta=0.1725
  k=16 -> silueta=0.1883
  k=20 -> silueta=0.2163
  k=24 -> silueta=0.2159
  k=8 -> silueta=0.1463
Comparacion de metodos (n=3.000):
  kmeans   -> silueta=0.2010
  ward     -> silueta=0.1464
  dbscan   -> silueta=-0.1448
  dbscan_noise -> silueta=0.2700
  ruido DBSCAN -> 27.0%


In [6]:
print("Ocho perfiles con mas vacantes")
print("-" * 52)
for n, (count, label) in enumerate(DATA["segments"], 1):
    pct = 100 * count / DATA["rows"]
    print(f"{n:2d}. {label:20s} {count:7,d} ({pct:4.1f}%)".replace(",", "."))

Ocho perfiles con mas vacantes
----------------------------------------------------
 1. Servicio al cliente  100.975 (18.1%)
 2. Gestion/Analista      88.565 (15.9%)
 3. Comercial             62.869 (11.3%)
 4. Ventas                43.059 ( 7.7%)
 5. Produccion            34.609 ( 6.2%)
 6. Construccion          31.114 ( 5.6%)
 7. Bodega                28.881 ( 5.2%)
 8. Call center           28.576 ( 5.1%)


In [7]:
s = DATA["salary"]
print("Clasificacion de banda salarial (SPE)")
print("-" * 52)
print(f"Vacantes etiquetadas ..... {s['labeled_rows']:,}".replace(",", "."))
print(f"Validacion cruzada ....... {s['cv_folds']} pliegues, {s['cv_rows']:,} filas".replace(",", "."))
print(f"Bandas ................... {len(s['bands'])}: {', '.join(s['bands'])}")
print(f"Exactitud ................ {s['accuracy']:.3f} (base {s['baseline_accuracy']:.3f})")
print(f"Macro-F1 ................. {s['macro_f1']:.3f} (base {s['baseline_macro_f1']:.3f})")
print(f"A 1 banda o menos ........ {s['within_one_band']:.3f} (base {s['baseline_within_one_band']:.3f})")

Clasificacion de banda salarial (SPE)
----------------------------------------------------
Vacantes etiquetadas ..... 431.505
Validacion cruzada ....... 5 pliegues. 100.000 filas
Bandas ................... 6: Hasta $1,5M, $1,5M – $2M, $2M – $3M, $3M – $4M, $4M – $6M, Más de $6M
Exactitud ................ 0.748 (base 0.604)
Macro-F1 ................. 0.563 (base 0.126)
A 1 banda o menos ........ 0.945 (base 0.856)


In [8]:
g = DATA["graph"]
print("Grafo de habilidades")
print("-" * 52)
print(f"Nodos de habilidades ..... {g['skill_nodes']}")
print(f"Aristas de co-ocurrencia . {g['skill_edges']:,}".replace(",", "."))
print(f"Comunidades .............. {g['communities']}")
print(f"Modularidad .............. {g['modularity']:.4f}")
print("\nHabilidades con mayor grado ponderado")
for item in DATA["top_weighted"][:3]:
    print(f"  {item['skill']:28s} {item['weighted_degree']:7,d} (postings={item['postings']:,})".replace(",", "."))


Grafo de habilidades
----------------------------------------------------
Nodos de habilidades ..... 400
Aristas de co-ocurrencia . 24.680
Comunidades .............. 6
Modularidad .............. 0.2775

Habilidades con mayor grado ponderado
  comunicación                 174.955 (postings=71.387)
  servicio al cliente          172.934 (postings=101.043)
  técnicas de venta            166.575 (postings=107.330)


In [9]:
print("Validacion disponible")
print("-" * 52)
print(f"Silueta coseno ............... {DATA['silhouette_sample_rows']:,} vacantes".replace(",", "."))
print(f"Comparacion de metodos ....... {DATA['comparison']['sample_rows']:,} vacantes".replace(",", "."))
print(f"Clasificacion de bandas ...... {DATA['salary']['cv_folds']} pliegues, {DATA['salary']['cv_rows']:,} filas".replace(",", "."))
print("Revision manual .............. no realizada: sin conteo ni acuerdos que reportar")
print("Propuesto .................... 200 menciones ESCO + 200 asignaciones de segmento")

Validacion disponible
----------------------------------------------------
Silueta coseno ............... 5.000 vacantes
Comparacion de metodos ....... 3.000 vacantes
Clasificacion de bandas ...... 5 pliegues. 100.000 filas
Revision manual .............. no realizada: sin conteo ni acuerdos que reportar
Propuesto .................... 200 menciones ESCO + 200 asignaciones de segmento


In [10]:
checks = [
    (DATA["rows"] == 557279, 'filas del corpus'),
    (DATA['text_rows'] == DATA['rows'], 'filas con texto'),
    (round(DATA["silhouette_full"], 4) == 0.015, 'silueta del paper'),
    (round(DATA["comparison"]["kmeans"], 4) == 0.201, 'silueta K-Means'),
    (round(DATA["comparison"]["ward"], 4) == 0.1464, 'silueta Ward'),
    (round(DATA["comparison"]["dbscan"], 4) == -0.1448, 'silueta DBSCAN'),
    (round(DATA["salary"]["accuracy"], 3) == 0.748, 'exactitud del paper'),
    (round(DATA["salary"]["baseline_accuracy"], 3) == 0.604, 'linea base'),
    (round(DATA["salary"]["within_one_band"], 3) == 0.945, 'banda adyacente'),
    (DATA["clusters"] == 16, 'numero de perfiles'),
    (len(DATA['top_weighted']) >= 3, 'grado ponderado disponible'),
]
for ok, label in checks:
    print(("OK   " if ok else "FALLA") + " " + label)
assert all(ok for ok, _ in checks), "Las cifras del notebook no coinciden con el paper"
print("Todas las cifras coinciden con el paper final.")

OK    filas del corpus
OK    filas con texto
OK    silueta del paper
OK    silueta K-Means
OK    silueta Ward
OK    silueta DBSCAN
OK    exactitud del paper
OK    linea base
OK    banda adyacente
OK    numero de perfiles
OK    grado ponderado disponible
Todas las cifras coinciden con el paper final.


## Interpretación

El corpus fijado tiene más de medio millón de vacantes, pero la silueta coseno es baja: los perfiles ordenan el mercado sin separarlo de forma nítida. La clasificación de bandas supera a la línea base, en especial en macro-F1 y en aciertos a una banda de distancia. El grado ponderado del grafo destaca habilidades que coocurren con muchas otras en numerosas vacantes, no solo las más frecuentes.

## Limitaciones

- Los valores son copias de los artefactos del run fijado; las aserciones comprueban coherencia interna con el paper, no contra Hugging Face.
- No se hizo revisión manual de muestra: no hay conteo de vacantes revisadas ni acuerdos de anotación.
- La validación manual de menciones ESCO, temas y grupos sigue pendiente.
- La coincidencia exacta con ESCO no resuelve contexto ni sinónimos locales.
- El corpus cubre las fuentes capturadas, no todo el empleo colombiano.

Los valores de este notebook son los que aparecen en las figuras y tablas del paper final.